# 02 — Model Development
## SkillAlign AI — Training Pipeline Lengkap
**Tim:** CC26-PSU318 | **Capstone Project** Coding Camp 2026

---
Notebook ini berisi pipeline lengkap:
1. Data Loading & Preparation
2. Text Preprocessing (NLP)
3. Word Embedding (Word2Vec)
4. Model Building (Dual-Input CNN + Custom Attention)
5. Training & Evaluation
6. Model Export

In [6]:
# ============================================
# 1. IMPORTS
# ============================================
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from sklearn.model_selection import train_test_split

from src.preprocessing.nlp_preprocessor import NLPPreprocessor
from src.preprocessing.embeddings import EmbeddingManager
from src.preprocessing.feature_engineering import FeatureEngineer
from src.models.model_architecture import SkillAlignMatcher
from src.models.custom_loss import focal_loss
from src.models.custom_callbacks import F1ScoreCallback
from src.training.train import ModelTrainer, TRAINING_CONFIG
from src.utils.metrics import compute_all_metrics, compute_classification_report, check_performance_targets
from src.utils.visualization import TrainingVisualizer, MetricsLogger

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('All imports successful!')

TensorFlow version: 2.21.0
GPU available: False
All imports successful!


## 2. Data Loading & Preparation

Dataset `datasets.csv` berisi data structured (skills, salary, industry, dll).
Untuk NLP model, kita juga membutuhkan `job_description` dari `job_posting.csv`.

**Strategi:** Kita akan membuat CV-Job pairs dengan matching labels
berdasarkan skill overlap antara simulated CV dan job requirements.

In [7]:
# Load datasets
df = pd.read_csv('../Dataset/ai_ready_datasets.csv')
print(f'datasets.csv shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

datasets.csv shape: (123842, 9)
Columns: ['title', 'job_description', 'formatted_experience_level', 'skills_desc', 'aggregated_skills', 'aggregated_benefits', 'aggregated_industries', 'company_name', 'aggregated_specialities']


,title,job_description,formatted_experience_level,skills_desc,aggregated_skills,aggregated_benefits,aggregated_industries,company_name,aggregated_specialities
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Mid-Senior level,Requirements: \n\nWe are seeking a College or ...,"Marketing, Sales",NaN,Real Estate,Corcoran Sawyer Smith,"real estate, new development"
1,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",Mid-Senior level,NaN,Health Care Provider,NaN,Unknown,Unknown Company,NaN
2,Assitant Restaurant Manager,The National Exemplar is accepting application...,Mid-Senior level,We are currently accepting resumes for FOH - A...,"Management, Manufacturing",NaN,Restaurants,The National Exemplar,NaN
3,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,Mid-Senior level,This position requires a baseline understandin...,Other,401(k),Law Practice,"Abrams Fensterman, LLP","Civil Litigation, Corporate & Securities Law, ..."
4,Service Technician,Looking for HVAC service tech with experience ...,Mid-Senior level,NaN,Information Technology,NaN,Facilities Services,Unknown Company,NaN


In [8]:
# Load job_posting untuk job_description dan skills_desc
df_posting = pd.read_csv(
    '../Dataset/database_design/job_posting.csv',
    usecols=['job_posting_id', 'title', 'job_description', 'skills_desc']
)
print(f'job_posting.csv shape: {df_posting.shape}')
# Drop rows tanpa job_description
df_posting = df_posting.dropna(subset=['job_description'])
print(f'After dropping nulls: {df_posting.shape}')
df_posting.head(3)

job_posting.csv shape: (123849, 4)
After dropping nulls: (123842, 4)


,job_posting_id,title,job_description,skills_desc
0,921716,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...
1,1829192,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",NaN
2,10998357,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...


In [9]:
# Load skills dictionary
df_skills = pd.read_csv('../Dataset/database_design/skills.csv')
skill_dictionary = df_skills['skill_name'].dropna().unique().tolist()
print(f'Total skills in dictionary: {len(skill_dictionary)}')
print(f'Sample skills: {skill_dictionary[:10]}')

Total skills in dictionary: 35
Sample skills: ['Art/Creative', 'Design', 'Advertising', 'Product Management', 'Distribution', 'Education', 'Training', 'Project Management', 'Consulting', 'Purchasing']


### 2.1 Generate CV-Job Pairs
Karena dataset asli adalah job postings (bukan CV-Job pairs),
kita perlu membuat synthetic matching data.

In [10]:
# Load job_skills untuk membuat synthetic CV
df_job_skills = pd.read_csv('../Dataset/database_design/job_skills.csv')
df_skill_names = pd.read_csv('../Dataset/database_design/skills.csv')
df_job_skills = df_job_skills.merge(df_skill_names, on='skill_id', how='left')

# Group skills per job
job_skills_grouped = df_job_skills.groupby('job_posting_id')['skill_name'].apply(list).reset_index()
job_skills_grouped.columns = ['job_posting_id', 'required_skills']

# Merge dengan job descriptions
df_pairs = df_posting.merge(job_skills_grouped, on='job_posting_id', how='inner')
df_pairs = df_pairs.dropna(subset=['job_description', 'required_skills'])
print(f'Jobs with descriptions + skills: {len(df_pairs):,}')
df_pairs.head(3)

Jobs with descriptions + skills: 122,090


,job_posting_id,title,job_description,skills_desc,required_skills
0,921716,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...,"[Marketing, Sales]"
1,1829192,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",NaN,[Health Care Provider]
2,10998357,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...,"[Management, Manufacturing]"


In [11]:
# Generate synthetic CV texts
# Strategi: untuk setiap job, buat CV yang match (overlap tinggi)
# dan CV yang tidak match (overlap rendah)
np.random.seed(42)

all_skills = df_skill_names['skill_name'].dropna().unique().tolist()

cv_texts = []
job_texts = []
labels = []

# Ambil sample jobs (sesuaikan jumlah berdasarkan kebutuhan)
sample_size = min(len(df_pairs), 20000)
df_sample = df_pairs.sample(n=sample_size, random_state=42).reset_index(drop=True)

for idx, row in df_sample.iterrows():
    job_desc = str(row['job_description'])
    req_skills = row['required_skills'] if isinstance(row['required_skills'], list) else []
    title = str(row.get('title', 'Professional'))
    
    # === POSITIVE PAIR (match) ===
    # CV yang memiliki overlap skill tinggi dengan job
    n_match = max(1, int(len(req_skills) * np.random.uniform(0.6, 1.0)))
    matched_skills = np.random.choice(req_skills, size=min(n_match, len(req_skills)), replace=False).tolist()
    cv_positive = f"Experienced {title} with skills in {', '.join(matched_skills)}. "
    cv_positive += f"Strong background in {matched_skills[0] if matched_skills else 'technology'}. "
    cv_positive += f"Proficient professional with {np.random.randint(1, 10)} years of experience."
    
    cv_texts.append(cv_positive)
    job_texts.append(job_desc[:2000])  # Truncate very long descriptions
    labels.append(1)
    
    # === NEGATIVE PAIR (not match) ===
    # CV dengan skill yang berbeda dari job requirements
    other_skills = [s for s in all_skills if s not in req_skills]
    if len(other_skills) >= 3:
        random_skills = np.random.choice(other_skills, size=min(5, len(other_skills)), replace=False).tolist()
    else:
        random_skills = np.random.choice(all_skills, size=3, replace=False).tolist()
    cv_negative = f"Professional with expertise in {', '.join(random_skills)}. "
    cv_negative += f"Background in {random_skills[0]}. {np.random.randint(1, 8)} years experience."
    
    cv_texts.append(cv_negative)
    job_texts.append(job_desc[:2000])
    labels.append(0)

labels = np.array(labels, dtype=np.float32)
print(f'Total pairs generated: {len(labels):,}')
print(f'Positive (match): {int(labels.sum()):,}')
print(f'Negative (not match): {int(len(labels) - labels.sum()):,}')
print(f'Balance ratio: {labels.mean():.2%}')

Total pairs generated: 40,000
Positive (match): 20,000
Negative (not match): 20,000
Balance ratio: 50.00%


## 3. Text Preprocessing

In [12]:
# Konfigurasi preprocessor
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 300  # Sesuaikan berdasarkan analisis panjang teks di EDA
EMBEDDING_DIM = 128

preprocessor = NLPPreprocessor(
    max_vocab_size=MAX_VOCAB_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    use_lemmatizer=True,
    language='english'
)

print('Preprocessor initialized.')
print(f'  Max vocab size: {MAX_VOCAB_SIZE}')
print(f'  Max sequence length: {MAX_SEQ_LEN}')

Preprocessor initialized.
  Max vocab size: 10000
  Max sequence length: 300


In [13]:
# Fit tokenizer pada semua teks (CV + Job)
all_texts = cv_texts + job_texts
preprocessor.fit(all_texts)
print(f'Vocabulary size: {preprocessor.vocab_size}')

Vocabulary size: 10000


In [14]:
# Transform ke sequences
cv_sequences = preprocessor.transform(cv_texts)
job_sequences = preprocessor.transform(job_texts)

print(f'CV sequences shape: {cv_sequences.shape}')
print(f'Job sequences shape: {job_sequences.shape}')

CV sequences shape: (40000, 300)
Job sequences shape: (40000, 300)


## 4. Word Embedding (Word2Vec)

In [15]:
# Train Word2Vec pada corpus
# Tokenize texts menjadi list of lists
tokenized_texts = [text.split() for text in preprocessor.preprocess_batch(all_texts)]

emb_manager = EmbeddingManager(embedding_dim=EMBEDDING_DIM, min_count=2, window=5)
emb_manager.train_word2vec(tokenized_texts, epochs=20, sg=1)
print('Word2Vec training complete.')

Word2Vec training complete.


In [16]:
# Buat embedding matrix
embedding_matrix = emb_manager.create_embedding_matrix(
    word_index=preprocessor.word_index,
    vocab_size=preprocessor.vocab_size
)
print(f'Embedding matrix shape: {embedding_matrix.shape}')

Embedding matrix shape: (10000, 128)


In [17]:
# Test similar words
test_words = ['python', 'data', 'manager', 'engineer']
for word in test_words:
    similar = emb_manager.get_similar_words(word, topn=5)
    if similar:
        print(f'\n"{word}" -> {[(w, round(s, 3)) for w, s in similar]}')


"python" -> [('golang', 0.752), ('java', 0.75), ('scala', 0.744), ('pig', 0.736), ('bash', 0.73)]

"data" -> [('analytics', 0.713), ('reportingextracts', 0.682), ('warehousesperformance', 0.648), ('availabilitycollaborate', 0.647), ('etl', 0.647)]

"manager" -> [('director', 0.725), ('bonused', 0.657), ('supervisor', 0.638), ('dyad', 0.618), ('keyholdersmaster', 0.616)]

"engineer" -> [('engineering', 0.699), ('experiencedsr', 0.698), ('architect', 0.696), ('taskrabbit', 0.659), ('foras', 0.65)]


## 5. Train-Test Split

In [18]:
# Stratified split
cv_train, cv_test, job_train, job_test, y_train, y_test = train_test_split(
    cv_sequences, job_sequences, labels,
    test_size=0.2, random_state=42, stratify=labels
)

# Split validation dari training
cv_train, cv_val, job_train, job_val, y_train, y_val = train_test_split(
    cv_train, job_train, y_train,
    test_size=0.15, random_state=42, stratify=y_train
)

print(f'Train: {len(y_train):,} (pos={y_train.sum():.0f}, neg={len(y_train)-y_train.sum():.0f})')
print(f'Val:   {len(y_val):,} (pos={y_val.sum():.0f}, neg={len(y_val)-y_val.sum():.0f})')
print(f'Test:  {len(y_test):,} (pos={y_test.sum():.0f}, neg={len(y_test)-y_test.sum():.0f})')

Train: 27,200 (pos=13600, neg=13600)
Val:   4,800 (pos=2400, neg=2400)
Test:  8,000 (pos=4000, neg=4000)


## 6. Build Model

In [19]:
# Build SkillAlign Matcher
matcher = SkillAlignMatcher(
    vocab_size=preprocessor.vocab_size,
    max_seq_len=MAX_SEQ_LEN,
    embedding_dim=EMBEDDING_DIM,
    attention_units=128,
    embedding_matrix=embedding_matrix,
    trainable_embedding=True
)

model = matcher.build_model()
model.summary()

Model: "SkillAlign_Matcher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cv_input            │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_input           │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 300, 128)  │  1,280,000 │ cv_input[0][0],   │
│ (Embedding)         │                   │            │ job_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_conv1d_1         │ (None, 300, 128)  │     49,280 │ shared_embedding… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_conv1d_1        │ (None, 300, 128)  │     49,280 │ shared_embedding… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_bn_1             │ (None, 300, 128)  │        512 │ cv_conv1d_1[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_bn_1            │ (None, 300, 128)  │        512 │ job_conv1d_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_conv1d_2         │ (None, 300, 64)   │     24,640 │ cv_bn_1[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_conv1d_2        │ (None, 300, 64)   │     24,640 │ job_bn_1[0][0]    │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_bn_2             │ (None, 300, 64)   │        256 │ cv_conv1d_2[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_bn_2            │ (None, 300, 64)   │        256 │ job_conv1d_2[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ custom_attention    │ (None, 128)       │     24,960 │ cv_bn_2[0][0],    │
│ (CustomAttentionLa… │                   │            │ job_bn_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_global_pool      │ (None, 64)        │          0 │ cv_bn_2[0][0]     │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_global_pool     │ (None, 64)        │          0 │ job_bn_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_merge       │ (None, 256)       │          0 │ custom_attention… │
│ (Concatenate)       │                   │            │ cv_global_pool[0… │
│                     │                   │            │ job_global_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ feature_merge[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_bn_1          │ (None, 256)       │      1,024 │ dense_1[0][0]   

 Total params: 1,562,881 (5.96 MB)

 Trainable params: 1,561,345 (5.96 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [20]:
# Visualisasi model architecture
tf.keras.utils.plot_model(
    model,
    to_file='../logs/model_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    dpi=100
)
print('Model architecture plot saved.')

You must install pydot (`pip install pydot`) for `plot_model` to work.
Model architecture plot saved.


## 7. Training

In [21]:
# Setup trainer
trainer = ModelTrainer(
    model=model,
    config={
        'batch_size': 32,
        'epochs': 50,
        'learning_rate': 0.001,
        'early_stopping_patience': 10,
        'reduce_lr_patience': 5,
        'f1_patience': 8,
        'f1_threshold': 0.80,
        'focal_loss_gamma': 2.0,
        'focal_loss_alpha': 0.25
    },
    log_dir='../logs/training',
    model_dir='../models'
)

# Compile
trainer.compile_model()
print('Model compiled. Ready for training.')

Model compiled. Ready for training.


In [22]:
# Train!
history = trainer.train(
    x_train=[cv_train, job_train],
    y_train=y_train,
    x_val=[cv_val, job_val],
    y_val=y_val
)
print('\nTraining complete!')

Epoch 1/50
849/850 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9769 - auc: 0.9916 - loss: 0.0114 - precision: 0.9736 - recall: 0.9823
Epoch 1: val_accuracy improved from None to 1.00000, saving model to ../models\best_model.keras

  [F1Callback] Epoch 1: F1=1.0000 | Precision=1.0000 | Recall=1.0000 | Best F1=0.0000
  [F1Callback] Model saved to ../models\best_f1_model.keras (F1=1.0000)

  [F1Callback] F1-score 1.0000 >= threshold 0.8000. Target tercapai!
850/850 ━━━━━━━━━━━━━━━━━━━━ 50s 54ms/step - accuracy: 0.9954 - auc: 0.9999 - loss: 0.0021 - precision: 0.9949 - recall: 0.9959 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 5.3121e-09 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 0.0010 - val_f1_score: 1.0000 - val_precision_custom: 1.0000 - val_recall_custom: 1.0000
Epoch 2/50
849/850 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9998 - auc: 1.0000 - loss: 3.0654e-05 - precision: 1.0000 - recall: 0.9996
Epoch 2: val_accuracy did not improve from 1.00000



## 8. Training Visualization

In [23]:
viz = TrainingVisualizer(save_dir='../logs/plots')
viz.plot_training_history(history)
print('Training history plots saved.')

Training history plots saved.


## 9. Evaluation

In [24]:
# Evaluate on test set
test_results = trainer.evaluate(
    x_test=[cv_test, job_test],
    y_test=y_test
)
print('\nTest Results:', test_results)

250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 1.0000 - auc: 1.0000 - loss: 5.7286e-13 - precision: 1.0000 - recall: 1.0000

Test Results: {'loss': 5.728576250516038e-13, 'compile_metrics': 1.0}


In [25]:
# Detailed metrics
y_pred_proba = model.predict([cv_test, job_test], verbose=0)
metrics = compute_all_metrics(y_test, y_pred_proba)
print('\n=== Detailed Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

# Classification report
print('\n=== Classification Report ===')
print(compute_classification_report(y_test, y_pred_proba))


=== Detailed Metrics ===
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1_score: 1.0
  mae: 0.0001
  auc: 1.0

=== Classification Report ===
              precision    recall  f1-score   support

   Not Match       1.00      1.00      1.00      4000
       Match       1.00      1.00      1.00      4000

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000



In [26]:
# Check performance targets
targets = check_performance_targets(metrics)
print('\n=== Performance Targets ===')
for name, info in targets.items():
    status = 'PASS' if info['passed'] else 'FAIL'
    print(f"  {name}: {info['value']} (target: {info['target']}) [{status}]")


=== Performance Targets ===
  accuracy: 1.0 (target: >= 0.85) [PASS]
  mae: 0.0001 (target: <= 0.02) [PASS]


In [27]:
# Confusion Matrix
y_pred_binary = (y_pred_proba > 0.5).astype(int).flatten()
viz.plot_confusion_matrix(y_test, y_pred_binary)

# Score distribution
viz.plot_score_distribution(y_pred_proba.flatten(), y_test)

# Metrics comparison
viz.plot_metrics_comparison(
    metrics,
    targets={'accuracy': 0.85, 'f1_score': 0.80, 'precision': 0.80, 'recall': 0.80}
)
print('All evaluation plots saved.')

All evaluation plots saved.


## 10. Save Model & Preprocessor

In [28]:
# Save final model
model_path = trainer.save_model('skillalign_matcher.keras')
print(f'Model saved: {model_path}')

# Save as SavedModel format too
savedmodel_path = os.path.join('..', 'models', 'saved_model')
tf.saved_model.save(model, savedmodel_path)
print(f'SavedModel saved: {savedmodel_path}')

Model saved: ../models\skillalign_matcher.keras
INFO:tensorflow:Assets written to: ..\models\saved_model\assets


INFO:tensorflow:Assets written to: ..\models\saved_model\assets


SavedModel saved: ..\models\saved_model


In [29]:
# Save preprocessor
preprocessor_path = '../preprocessors/nlp_preprocessor.pkl'
os.makedirs(os.path.dirname(preprocessor_path), exist_ok=True)
joblib.dump(preprocessor, preprocessor_path)
print(f'Preprocessor saved: {preprocessor_path}')

# Save embedding manager
emb_path = '../preprocessors/embedding_manager.pkl'
emb_manager.save_model(emb_path)
print(f'Embedding manager saved: {emb_path}')

Preprocessor saved: ../preprocessors/nlp_preprocessor.pkl
Embedding manager saved: ../preprocessors/embedding_manager.pkl


In [30]:
# Save model config
import json
config = matcher.get_model_config()
config['training_summary'] = trainer.get_training_summary()
config['evaluation_metrics'] = metrics
config_path = '../models/model_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f'Model config saved: {config_path}')
print('\n=== All artifacts saved successfully! ===')

Model config saved: ../models/model_config.json

=== All artifacts saved successfully! ===


## 11. Quick Inference Test

In [31]:
# Test inference
from src.inference.predict import SkillAlignPredictor

predictor = SkillAlignPredictor(
    model_path='../models/skillalign_matcher.keras',
    preprocessor_path='../preprocessors/nlp_preprocessor.pkl'
)
predictor.load()

# Test case
result = predictor.predict(
    cv_text='Experienced Data Scientist with 5 years in Python, TensorFlow, machine learning, deep learning, and data analysis.',
    job_description='Looking for a Data Scientist with strong Python skills, experience in ML frameworks, TensorFlow, and statistical analysis.'
)

print(f'Matching Score: {result.matching_score}')
print(f'Confidence: {result.confidence}')
print(f'Recommendation: {result.recommendation}')
print(f'Inference Time: {result.inference_time_ms:.1f}ms')

Matching Score: 0.0709
Confidence: Low
Recommendation: Not Recommended
Inference Time: 280.5ms
